In [1]:
import numpy as np
import matplotlib.pyplot as plt

In [9]:
import numpy as np
from tensorflow.keras.datasets import fashion_mnist

# =========================
# Load Fashion MNIST
# =========================
(X_train, y_train), (X_test, y_test) = fashion_mnist.load_data()

# Use SMALL subset to FORCE overfitting
X_train = X_train[:1000]
y_train = y_train[:1000]
X_test = X_test[:1000]
y_test = y_test[:1000]

# Normalize
X_train = X_train.reshape(len(X_train), -1) / 255.0
X_test = X_test.reshape(len(X_test), -1) / 255.0

# One-hot labels
def one_hot(y):
    out = np.zeros((len(y), 10))
    out[np.arange(len(y)), y] = 1
    return out

y_train_oh = one_hot(y_train)
y_test_oh = one_hot(y_test)

# =========================
# Activations
# =========================
def relu(x):
    return np.maximum(0, x)

def relu_deriv(x):
    return (x > 0).astype(float)

def softmax(z):
    e = np.exp(z - np.max(z, axis=1, keepdims=True))
    return e / np.sum(e, axis=1, keepdims=True)

# =========================
# Loss
# =========================
def cross_entropy(y, yhat):
    return -np.mean(np.sum(y * np.log(yhat + 1e-8), axis=1))

# =========================
# Dropout
# =========================
def dropout(a, rate):
    mask = (np.random.rand(*a.shape) > rate)
    return a * mask / (1-rate), mask

# =========================
# Accuracy
# =========================
def accuracy(y, yhat):
    return np.mean(np.argmax(y,1)==np.argmax(yhat,1))

# =========================
# Network
# =========================
np.random.seed(0)

D = 784
H = 512
C = 10
lr = 0.05
epochs = 1000

W1 = np.random.randn(D,H)*0.01
b1 = np.zeros((1,H))
W2 = np.random.randn(H,C)*0.01
b2 = np.zeros((1,C))

# =========================
# Training function
# =========================
def train(use_reg=False, lam=0.001, drop=0.3):

    global W1,W2,b1,b2

    for e in range(epochs):

        # Forward
        z1 = X_train @ W1 + b1
        a1 = relu(z1)

        if use_reg:
            a1, mask = dropout(a1, drop)

        z2 = a1 @ W2 + b2
        out = softmax(z2)

        loss = cross_entropy(y_train_oh,out)

        if use_reg:
            loss += lam*(np.sum(W1*W1)+np.sum(W2*W2))

        # Backprop
        dz2 = out - y_train_oh
        dW2 = a1.T @ dz2 / len(X_train)
        db2 = np.mean(dz2,0,keepdims=True)

        da1 = dz2 @ W2.T
        if use_reg:
            da1 *= mask/(1-drop)

        dz1 = da1 * relu_deriv(z1)
        dW1 = X_train.T @ dz1 / len(X_train)
        db1 = np.mean(dz1,0,keepdims=True)

        if use_reg:
            dW1 += lam*W1
            dW2 += lam*W2

        # Update
        W1 -= lr*dW1
        b1 -= lr*db1
        W2 -= lr*dW2
        b2 -= lr*db2

        if e%500==0:
            print("Epoch",e,"Loss",round(loss,3))

    # Final accuracy
    train_pred = softmax(relu(X_train@W1+b1)@W2+b2)
    test_pred = softmax(relu(X_test@W1+b1)@W2+b2)

    print("\nTrain:",accuracy(y_train_oh,train_pred))
    print("Val:",accuracy(y_test_oh,test_pred))

# =========================
# WITHOUT REGULARIZATION
# =========================
print("\nWITHOUT REGULARIZATION")
train(False)

# Reset weights
W1 = np.random.randn(D,H)*0.01
b1 = np.zeros((1,H))
W2 = np.random.randn(H,C)*0.01
b2 = np.zeros((1,C))

# =========================
# WITH DROPOUT + L2
# =========================
print("\nWITH DROPOUT + L2")
train(True, lam=0.01, drop=0.5)



WITHOUT REGULARIZATION
Epoch 0 Loss 2.306
Epoch 500 Loss 0.462

Train: 0.917
Val: 0.789

WITH DROPOUT + L2
Epoch 0 Loss 2.71
Epoch 500 Loss 1.037

Train: 0.895
Val: 0.789
